> **Series**: *Bayesian Modelling for Industrial Applications*, Part 1. [Part 2](https://sambaiga.github.io/pages/blog/posts/2025/12/2025-12-1-bayesian-regression..html) fits a Beta regression to battery degradation, and [Part 3](https://sambaiga.github.io/pages/blog/posts/2026/11/2026-11-21-hierarchical-bayesian-regression.html) lets each cell and chemistry have its own curve.

## A Decision with Little Data

A plant has installed twelve identical machines on a new line. The lead engineer, drawing on earlier models, expects a machine to last about ten years. After a three-month pilot of the whole line, one machine has failed, on day 40, and the other eleven have run without a fault. The supplier now offers a three-year service contract at 20 thousand euros per machine, and a repair costs 60 thousand euros. The contract has to be decided before more data exist.

This is the situation of most industrial decisions: there is expert knowledge, there is a little evidence, and neither is enough alone. Bayesian inference combines them and says how uncertain the combination is. The post follows one problem from the model to the decision. It builds the model, fits it with PyMC,[^pymc] checks the fit against an exact answer, and then asks the question that matters for the contract: does the decision survive a change of the prior? The costs are illustrative.

[^pymc]: O. Abril-Pla *et al.*, "PyMC: A modern, and comprehensive probabilistic programming framework in Python," *PeerJ Comput. Sci.*, vol. 9, p. e1516, 2023.

## The Building Blocks of Bayesian Modelling

Frequentist methods draw their confidence from a large number of repeated trials, a luxury that a new machine does not offer. Bayesian inference instead treats the unknown quantity, here the failure rate $\theta$ of a machine, as a random variable with a distribution that expresses what is believed about it. The belief is updated by Bayes' theorem,[^bda]

$$
p(\theta \mid D) = \frac{p(D \mid \theta)\, p(\theta)}{p(D)} \;\propto\; p(D \mid \theta)\, p(\theta),
$$

where $D$ is the pilot data. Three parts do the work.

1. The **prior** $p(\theta)$ is the belief before the data. The engineer's ten-year expectation is a prior.
2. The **likelihood** $p(D \mid \theta)$ says how probable the pilot is for each value of the rate. A rate of ten failures per machine-year makes a pilot with a single failure very unlikely, and so does a rate of one failure per thousand years.
3. The **posterior** $p(\theta \mid D)$ is the updated belief. It lies between the prior and the evidence, closer to whichever is stronger.

The evidence $p(D)$ only normalises the posterior, so it is not needed to compare values of $\theta$. Computing the normalised posterior is hard in general, because it is an integral over all parameters, and that is the problem that a probabilistic programming language such as PyMC solves with Markov chain Monte Carlo (MCMC). Alternatives in Python are Pyro and TensorFlow Probability. This post uses PyMC because its models read like the mathematics.

[^bda]: A. Gelman, J. B. Carlin, H. S. Stern, D. B. Dunson, A. Vehtari, and D. B. Rubin, *Bayesian Data Analysis*, 3rd ed. Boca Raton, FL, USA: CRC Press, 2013.

## The Pilot and Its Model

A machine is taken to fail at a constant rate $\theta$ per machine-year, so that its lifetime is exponential with mean $1/\theta$.[^meeker] The rate is constant for simplicity, and the assumption fails for a machine that wears out, a limit that is discussed at the end. The pilot ran $n = 12$ machines for 91 days, and the failed machine stopped on day 40, so the total exposure is $T = (11 \times 91 + 40)/365.25 = 2.85$ machine-years. The number of failures $d$ in the pilot is Poisson with mean $\theta T$,

$$
d \mid \theta \sim \mathrm{Poisson}(\theta T), \qquad \theta \sim \mathrm{Gamma}(a, b),
$$

where the Gamma prior has shape $a$ and rate $b$, so that its mean is $a/b$. This pair is conjugate: the posterior is again a Gamma, with shape $a + d$ and rate $b + T$. The update has a plain reading. The prior behaves like $a$ failures seen over $b$ machine-years, and the pilot adds $d$ failures over $T$ machine-years. The rate $b$ is therefore the weight of the prior in machine-years, which is the number to compare with the 2.85 machine-years of the pilot.

The engineer's expectation of ten years is a prior mean rate of 0.1 per machine-year. The prior Gamma(4, 40) has that mean, a standard deviation of 0.05, and the weight of 40 machine-years of experience. Two other priors are used later to test the decision: a pessimist, Gamma(4, 12), who expects a life of three years, and a weak prior, Gamma(1, 1), that carries almost no information.

[^meeker]: W. Q. Meeker and L. A. Escobar, *Statistical Methods for Reliability Data*. New York, NY, USA: Wiley, 1998.

In [ ]:
# | output: false
from importlib.metadata import version
import os
import warnings

warnings.filterwarnings("ignore")
os.environ.setdefault("OMP_NUM_THREADS", "2")

import arviz as az
import numpy as np
import pandas as pd
import pymc as pm
from scipy import stats

from bayes import lifetime
from bayes.plot import lifetime as figures
from bayes.plot.inline import show

FAST = os.environ.get("BAYES_FAST") == "1"  # a smoke test with few draws, to check that the notebook runs
DRAWS, TUNE, CHAINS = (200, 200, 2) if FAST else (2000, 1000, 4)
MACHINES, DAYS, FAILURE_DAYS = 12, 91, [40]  # the pilot: twelve machines for three months, one failure on day 40
FAILURES = len(FAILURE_DAYS)
EXPOSURE = lifetime.exposure_years(MACHINES, DAYS, FAILURE_DAYS)
PRIORS = {
    "Pessimist": (4.0, 12.0),
    "Engineer": (4.0, 40.0),
    "Weak": (1.0, 1.0),
}  # Gamma(shape, rate) on the rate per machine-year
CONTRACT = {"years": 3.0, "repair": 60.0, "price": 20.0}  # illustrative costs in thousands of euros
SHAPE, RATE = PRIORS["Engineer"]
generator = np.random.default_rng(2026)
print("pymc", version("pymc"), "| arviz", version("arviz"), "| fast mode:", FAST)
print(f"pilot: {MACHINES} machines, {DAYS} days, {FAILURES} failure, {EXPOSURE:.3f} machine-years")

## Checking the Prior before the Data

A prior predictive check asks what the prior implies about the data before they are seen. If the pilot looks impossible under the prior, either the prior or the model is wrong. @fig-prior shows the share of prior predictive pilots with each number of failures.

In [ ]:
# | label: fig-prior
# | fig-cap: "Share of prior predictive pilots with each number of failures, under the engineer's prior. The observed single failure (orange) has a probability of about one in five, so the pilot does not contradict the prior."
shares = lifetime.prior_failures(SHAPE, RATE, EXPOSURE, upper=6, generator=generator)
show(figures.prior_check(shares, FAILURES))

Under the engineer's prior, 76.1% of pilots show no failure, 20.3% show one and 3.1% show two. The observed single failure is therefore not surprising. It has a probability of about one in five, so the pilot does not contradict the engineer, and the prior is plausible enough to proceed.

## Fitting the Model with PyMC

The model has one parameter, and its posterior is known exactly, so the sampler is not needed for the answer. It is used here for a different reason, which is that a sampler can be checked against an exact result before it is trusted on a model that has none. The PyMC model states the prior and the likelihood, and the No-U-Turn sampler[^nuts] draws from the posterior with four chains of 2,000 draws after 1,000 tuning draws.

[^nuts]: M. D. Hoffman and A. Gelman, "The No-U-Turn Sampler: Adaptively setting path lengths in Hamiltonian Monte Carlo," *J. Mach. Learn. Res.*, vol. 15, pp. 1593-1623, 2014.

In [ ]:
# | output: false
model = lifetime.failure_model(SHAPE, RATE, FAILURES, EXPOSURE)
with model:
    idata = pm.sample(DRAWS, tune=TUNE, chains=CHAINS, cores=min(CHAINS, 2), random_seed=2026, progressbar=False)
post_shape, post_rate = lifetime.gamma_posterior(SHAPE, RATE, FAILURES, EXPOSURE)
draws = idata.posterior["theta"].to_numpy()
summary = az.summary(idata, hdi_prob=0.9)
print(summary[["mean", "sd", "hdi_5%", "hdi_95%", "ess_bulk", "r_hat"]].round(4).to_string())
print(
    f"exact Gamma({post_shape:g}, {post_rate:.2f}): mean {post_shape / post_rate:.4f}, sd {np.sqrt(post_shape) / post_rate:.4f}"
)

### Diagnostics

A sampler can fail without an error, so it is checked before its output is used. The first check is $\hat{R}$, the disagreement between chains, which should be close to one.[^rhat] The second is the effective sample size, the number of independent draws that the correlated chain is worth. The third is the number of divergent transitions, which should be zero. Here $\hat{R}$ is 1.00, the bulk and tail effective sample sizes are 3,513 and 4,367, and there is no divergence. The trace of @fig-trace shows the chains overlapping and moving freely around one level, which is what a healthy chain looks like.

The sampler also agrees with the exact answer. The posterior mean of the rate is 0.117 and its standard deviation 0.052 in the sample, against 0.1167 and 0.0522 for the exact Gamma(5, 42.85). The probability of a failure within three years is 0.2872 from the draws and 0.2871 from the closed form.

[^rhat]: A. Vehtari, A. Gelman, D. Simpson, B. Carpenter, and P.-C. Bürkner, "Rank-normalization, folding, and localization: An improved $\hat{R}$ for assessing convergence of MCMC," *Bayesian Anal.*, vol. 16, no. 2, pp. 667-718, 2021.

In [ ]:
# | label: fig-trace
# | fig-cap: "Draws of the failure rate in the four chains over the first 300 iterations. The chains overlap and move freely around one level, which is what converged sampling looks like."
show(figures.traces(draws[:, :300]))

## From Prior to Posterior

@fig-rates draws the three parts of Bayes' theorem on one axis. The likelihood is scaled to the height of the posterior so that it can share the axis.

In [ ]:
# | label: fig-rates
# | fig-cap: "Prior, likelihood and posterior of the failure rate. The likelihood (dashed) is scaled to the height of the posterior. The posterior stays near the prior because the prior is worth 40 machine-years and the pilot 2.85."
grid = np.linspace(0.002, 0.75, 400)
prior_density = stats.gamma.pdf(grid, SHAPE, scale=1 / RATE)
posterior_density = stats.gamma.pdf(grid, post_shape, scale=1 / post_rate)
likelihood = stats.gamma.pdf(grid, FAILURES + 1, scale=1 / EXPOSURE)
likelihood = (
    likelihood * posterior_density.max() / likelihood.max()
)  # scaled to the height of the posterior to share the axis
show(figures.rates(grid, prior_density, likelihood, posterior_density))

The likelihood alone peaks at 0.35 failures per machine-year, a mean life of 2.85 years, which is the estimate of the pilot with no prior. The prior peaks at 0.075. The posterior sits close to the prior, with a mean rate of 0.117 and a 90% interval from 0.036 to 0.196. The reason is the weights. The prior is worth 40 machine-years and the pilot 2.85, so the pilot moves the belief only a little. In years of life, the posterior median is 9.2, with a 90% interval from 4.7 to 21.8. A ten-year expectation has survived a pilot that, taken alone, suggests three.

Two cautions follow. The result depends on the prior, because the data are too few to overrule it. And one failure in a pilot is itself weak evidence: the repeat pilot of the same size would show no failure in 72.9% of cases and one in 22.2%.

## From the Posterior to the Contract

A posterior is useful when it changes a decision. The contract is worth buying when the repairs that it covers cost more than its price. A machine that fails at rate $\theta$ needs $3\theta$ repairs in three years, at 60 thousand euros each, so the expected cost without the contract is $180\,\theta$ thousand euros. That equals the price of 20 when $\theta = 0.111$, a mean life of 9.0 years. A machine that is expected to last less than nine years makes the contract worth buying.

The posterior turns that threshold into numbers. The expected cost of repairs without the contract is $180$ times the posterior mean rate, and the probability that repairs cost more than the price is the posterior probability that $\theta$ exceeds 0.111. The table gives these quantities for the three priors, and @fig-survival and @fig-decision show them.

In [ ]:
rows, theta_of = [], {}
for name, (shape, rate) in PRIORS.items():
    post = lifetime.gamma_posterior(shape, rate, FAILURES, EXPOSURE)
    theta = generator.gamma(post[0], 1.0 / post[1], 40000)
    theta_of[name] = theta
    low, mid, high = lifetime.lifetime_interval(theta)
    cost, probability = lifetime.contract_worth(theta, **CONTRACT)
    rows.append(
        {
            "prior": name,
            "prior mean life (y)": round(rate / shape, 1),
            "prior worth (machine-years)": rate,
            "posterior median life (y)": round(mid, 1),
            "90% interval (y)": f"{low:.1f} to {high:.1f}",
            "fails within 1 y": round(lifetime.failure_probability(theta, 1.0), 3),
            "fails within 3 y": round(lifetime.failure_probability(theta, 3.0), 3),
            "expected repairs (k EUR)": round(cost, 1),
            "P(repairs cost more)": round(probability, 2),
            "cost": cost,
            "probability": probability,
        }
    )
table = pd.DataFrame(rows)
display(table.drop(columns=["cost", "probability"]))

With the engineer's prior, the median life is 9.2 years, a machine fails within three years with probability 0.287, the expected cost of repairs is 21.0 thousand euros, and there is a 48% chance that repairs cost more than the contract. That is a toss-up. With the pessimist's prior the median life is 3.2 years, the probability of failing within three years is 0.601, the expected repairs cost 60.6, and the chance that they exceed the price is 97%. With the weak prior the median life is 2.3 years, the probability is 0.686, the cost is 94.0, and the chance is 93%.

In [ ]:
# | label: fig-survival
# | fig-cap: "Probability that a machine survives each time in service, under three priors. The dotted line marks the three-year contract. The engineer's prior (blue) gives 71% at three years, the other two about 40% and 32%."
years = np.linspace(0, 10, 101)
curves = pd.concat(
    [
        pd.DataFrame({"years": years, "survival": lifetime.survival(theta_of[name], years), "prior": name})
        for name in PRIORS
    ],
    ignore_index=True,
)
show(figures.survival_curves(curves, horizon=CONTRACT["years"], subject="Engineer"))

@fig-survival shows the same result as survival. At the three-year mark, about 71% of machines survive under the engineer's prior, and about 40% and 32% under the pessimist's and the weak prior.

In [ ]:
# | label: fig-decision
# | fig-cap: "Expected cost of repairs without the contract and the probability that repairs cost more than the contract price, for each prior. The dotted line is the price of 20. Only the engineer's prior makes the contract a toss-up."
show(figures.sensitivity(table[["prior", "cost", "probability"]], price=CONTRACT["price"]))

## What the Decision Depends On

The decision is not stable. The engineer's prior makes the contract a toss-up, at an expected cost of 21.0 against a price of 20, and the other two priors make it a clear purchase, with repairs expected to cost three to five times the price. The pilot is too short to settle the matter, because 2.85 machine-years of evidence cannot outweigh a prior that is worth 40, and it moves a prior that is worth one machine-year all the way to a life of 2.3 years.

That is the useful result of a Bayesian analysis, and it reports more than a single number does. It says that the contract is worth buying unless the engineer's ten-year expectation is right, and that more data are the way to find out. Each failure-free machine-year pushes the engineer's posterior further towards a long life, and each failure pushes it the other way, so the plant can extend the pilot and watch the 48% change.

In [ ]:
# | output: false
# Numbers that the text quotes. The cell prints them and is hidden from the post.
print("mle rate per year:", FAILURES / EXPOSURE, "| mle life:", EXPOSURE / FAILURES)
print("prior predictive shares of 0..6 failures:", shares.round(3))
print("ess/rhat:", summary[["ess_bulk", "ess_tail", "r_hat"]].round(3).to_dict("records"))
print("divergences:", int(idata.sample_stats["diverging"].sum()))
for name, (shape, rate) in PRIORS.items():
    post = lifetime.gamma_posterior(shape, rate, FAILURES, EXPOSURE)
    print(
        name,
        "posterior",
        post,
        "exact P(fail 3y):",
        round(lifetime.exact_probability(post[0], post[1], 3.0), 4),
        "exact P(cost>price):",
        round(
            float(
                stats.gamma.sf(CONTRACT["price"] / (CONTRACT["repair"] * CONTRACT["years"]), post[0], scale=1 / post[1])
            ),
            4,
        ),
        "exact mean rate:",
        round(post[0] / post[1], 4),
    )
print(
    "engineer MCMC vs exact P(fail 3y):",
    round(lifetime.failure_probability(draws.ravel(), 3.0), 4),
    round(lifetime.exact_probability(post_shape, post_rate, 3.0), 4),
)
print("break-even rate:", CONTRACT["price"] / (CONTRACT["repair"] * CONTRACT["years"]))
print(table.round(3).to_string())
posterior_shares = lifetime.prior_failures(post_shape, post_rate, EXPOSURE, upper=4, generator=generator)
print("posterior predictive shares of 0..4 failures in a repeat pilot:", posterior_shares.round(3))
print("break-even mean life (years):", 1 / (CONTRACT["price"] / (CONTRACT["repair"] * CONTRACT["years"])))
print("prior worth:", {name: rate for name, (shape, rate) in PRIORS.items()})

## Common Pitfalls and Good Practice

- **Report the prior.** A posterior is meaningless without its prior, and the sensitivity of the result to the prior is part of the answer, as the table shows.
- **State the weight of the prior.** For a Gamma prior the rate is the number of machine-years that it is worth, which is the quantity to compare with the evidence.
- **Check the prior with data not yet seen.** A prior predictive check catches a prior that makes the observed data implausible before any model is fitted.
- **Check the sampler.** $\hat{R}$, effective sample size and divergences take little time, and an exact result, where one exists, is a stronger check.
- **Question the likelihood.** A constant rate ignores wear-out. If the failures of a machine become more frequent with age, a Weibull lifetime fits better, and the conclusions about three years would change.
- **Turn the posterior into a decision.** An interval for a rate does not buy a contract. An expected cost and the probability that a threshold is crossed do.

## Next Steps in the Series

This post learned one number, a failure rate, from a short pilot. Part 2 moves to a continuous quantity, the state of health of a battery, and fits a regression with PyMC to six cells of one chemistry, testing it by predicting each cell from the other five. Part 3 then lets every cell and chemistry have its own curve, and tests what that gains on cells that the model has not seen.

## Reproducing the Results

The notebook is in the [bayesian-modelling](https://github.com/sambaiga/bayesian-modelling) repository and runs from the first cell to the last in the `uv` environment of the repository, with PyMC 5.25, ArviZ 0.22 and lets-plot 4.11. The model, the exact results and the figures are functions of the `bayes` package, which has unit tests, and the cells of the notebook only call them. Setting the environment variable `BAYES_FAST=1` runs the notebook with a few draws, to check that it works before the full run.